# z625 - Regresion Lineal (rearmado desde z403, para VM/Jupyter)

## Que hace este notebook
Es el script original `z403_RegresionLineal.ipynb` (que dio 0.231 en Kaggle), adaptado de Colab a tu entorno de Jupyter en la VM. La logica NO cambia, solo las rutas.

Resumen del metodo (para que quede documentado, no es mio, es el original):
- Lags: `tn_0` (actual) hasta `tn_11` (11 meses atras), mas `clase` = `tn` de 2 meses adelante (el target).
- Entrena SOLO con periodo 201812 y una lista curada de ~180 "productos magicos" (hardcodeada en el notebook original).
- Modelo: OLS (`statsmodels`) simple, con intercepto.
- Predice sobre 201912 para los 656 productos que tienen historial completo (12 meses); los 124 restantes se completan con el promedio del 2019.

**Ajustar `PARAM['datasets_path']` y `PARAM['exp_path']` a tus rutas reales antes de correr** (mismo problema de rutas que tuvimos en toda la materia -- confirmalas con `find` si hace falta).

In [1]:
!pip install -q polars statsmodels

In [2]:
import os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'LR01',
    'kaggle_competition': 'labo-iii-2026-ba',
    'semilla_primigenia': 102191,
    'empiojar_ruido': 0.0,
    'datasets_path': '/home/ds/datasets/',
    'exp_path': '/home/ds/exp/'
}

ruta = os.path.join(PARAM['exp_path'], PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LR01


## 1.2 Preprocesamiento

In [4]:
dataset = pl.read_csv(os.path.join(PARAM['datasets_path'], 'sell-in.txt.gz'), separator="\t")

In [5]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [6]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv(os.path.join(PARAM['datasets_path'], 'product_id_apredecir201912.txt'), separator="\t")

print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner")
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


### 1.2.1 Empiojado lognormal (desactivado por default, empiojar_ruido=0.0)

In [7]:
if PARAM['empiojar_ruido'] > 0.0:
    np.random.seed(PARAM['semilla_primigenia'])
    tb_ventas = tb_ventas.sort(["product_id", "periodo"])
    noise_multiplier = np.random.lognormal(mean=0.0, sigma=PARAM['empiojar_ruido'], size=tb_ventas.height)
    tb_ventas = tb_ventas.with_columns(
        (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
    )

### 1.2.2 Dataset aplanado con lags

In [8]:
lags = [-2, *range(0, 12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
    .with_columns(
        [
            pl.col("tn").shift(lag).over("product_id").alias(f"tn_{lag}")
            for lag in lags
        ]
    )
)

tb_lags = tb_lags.rename({"tn_-2": "clase"})

### 1.2.4 Definicion de Training -- productos magicos (lista original, hardcodeada)

In [9]:
# lista final (la segunda del notebook original pisa a la primera, asi que es la que se usa)
productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]

In [10]:
# Entreno con los datos de 2018 para los productos_magicos
dtrain = tb_lags.filter( (pl.col("periodo") == 201812) & (pl.col("product_id").is_in(productos_magicos)) )
print(dtrain.shape)

(182, 16)


## 1.3 Modelo de Regresion Lineal (OLS)

In [11]:
campos_buenos = dtrain.select(cs.starts_with("tn_"))

X_train = dtrain.select(campos_buenos).to_pandas()
X_train = sm.add_constant(X_train)
y_train = dtrain['clase'].to_pandas()

modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  clase   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     1336.
Date:                Sat, 15 Aug 2026   Prob (F-statistic):          1.41e-160
Time:                        14:51:37   Log-Likelihood:                -730.19
No. Observations:                 182   AIC:                             1486.
Df Residuals:                     169   BIC:                             1528.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0706      1.259      0.056      0.9

## 1.4 Aplicacion a los datos del futuro
Solo se puede aplicar a productos con todos los meses de 2019 completos (656 de 780); los otros 124 se completan con el promedio del 2019.

In [12]:
dfuture = tb_lags.filter( (pl.col("periodo") == 201912) & (pl.col("tn_11").is_not_null()) )
print(dfuture.shape)

(656, 16)


In [13]:
campos_buenos = dfuture.select(cs.starts_with("tn_"))
X_future = dfuture.select(campos_buenos).to_pandas()
X_future = sm.add_constant(X_future)

prediccion = modelo.predict(X_future)

tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)
print(tb_regresion.shape)

(656, 2)


### 1.4.3 Join de los modelos de regresion y promedio (fallback para los 124 restantes)

In [14]:
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter(pl.col("periodo").is_between(primer_periodo, ultimo_periodo)).group_by("product_id").agg(
    pl.col("tn").mean().alias("tn")
)
tb_meses12 = tb_meses12.select(["product_id", "tn"])

In [15]:
tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left", suffix="_update")
    .with_columns(
        pl.coalesce([pl.col("tn_pred"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_pred")
)
print(tb_final.shape)
tb_final.head()

(780, 2)


product_id,tn
i64,f64
20001,1124.297853
20002,1111.127719
20003,735.277478
20004,575.581884
20005,504.557617


## 1.5 Submit a Kaggle

In [16]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

if PARAM['empiojar_ruido'] <= 0.0:
    archivo = os.path.join(ruta, "linreg.csv")
    mensaje = "Regresion Lineal"
else:
    archivo = os.path.join(ruta, "linreg_empiojado.csv")
    mensaje = "Linear Regression logEMPIOJADO al " + str(PARAM['empiojar_ruido'])

tb_final.write_csv(archivo)
print(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

/home/ds/exp/LR01/linreg.csv


100%|██████████| 18.6k/18.6k [00:00<00:00, 57.1kB/s]


95 submissions remaining today.
Successfully submitted to Labo III, 2026 BA